# شبیه‌سازی سنگین نانومیله‌ی طلا (Meep، مختصات استوانه‌ای — فیزیک کاملاً سه‌بعدی)

**ترتیب اجرا:** سلول‌ها را از بالا به پایین اجرا کنید. سلول ۱ کرنل را یک‌بار ری‌استارت می‌کند (طبیعی است)؛ بعد از ری‌استارت از سلول ۲ ادامه دهید.

- نتایج مستقیم در Google Drive در پوشه‌ی `MyDrive/maghale_meep` ذخیره می‌شوند. اگر کولب قطع شد، فقط دوباره از بالا اجرا کنید: اجراهای تمام‌شده دوباره انجام نمی‌شوند.
- در پایان، سلول آخر یک فایل `results.zip` می‌سازد؛ آن را دانلود و در گفتگو برای Claude بفرستید (یا در مخزن در `isi/results/` بگذارید).
- **کولب رایگان** (۲ هسته): مراحل A و B حدود ۲ تا ۴ ساعت. مرحله‌ی C (نقشه‌ها) طولانی‌تر است و می‌توانید آن را در چند نشست اجرا کنید.


In [ ]:
# 1) نصب conda در کولب (کرنل یک‌بار ری‌استارت می‌شود)
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# 2) نصب Meep (حدود ۳ تا ۵ دقیقه)
!mamba install -q -y -c conda-forge "pymeep=*=mpi_mpich_*" h5py > /dev/null && echo "Meep installed"
import meep as mp; print('Meep', mp.__version__)

In [ ]:
# 3) اتصال Google Drive برای ذخیره‌ی پایدار نتایج
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/maghale_meep'; os.makedirs(OUT, exist_ok=True)
os.makedirs('/content/sim', exist_ok=True); os.chdir('/content/sim'); print('results ->', OUT)

In [ ]:
%%writefile /content/sim/gold_jc_params.json
{
 "einf": 2.2291060429787986,
 "f0": 8.275606537868544,
 "g0": 0.04903878814188577,
 "s0": 0.7012573060468409,
 "f1": 2.2076854977417635,
 "g1": 0.5088487040626065,
 "s1": 1.1558579136994516,
 "f2": 2.7550658553174983,
 "g2": 0.010000000000000038,
 "s2": 2.101031249400925
}

In [ ]:
%%writefile /content/sim/gold_jc.py
"""Johnson-Christy gold as a Meep Drude-Lorentz medium (fit 440-1000 nm, rms 1.6 %, see fit_jc.py).
Meep units: length 1 um, frequency f = 1/lambda[um]."""
import json, os, numpy as np
import meep as mp
_p = json.load(open(os.path.join(os.path.dirname(__file__), 'gold_jc_params.json')))

def eps_jc(lam_nm):
    """Permittivity of the fitted model (same as the Meep medium), lam in nm."""
    f = 1000.0 / np.asarray(lam_nm, float)
    p = _p
    return (p['einf'] - p['s0'] * p['f0']**2 / (f**2 + 1j * f * p['g0'])
            + p['s1'] * p['f1']**2 / (p['f1']**2 - f**2 - 1j * f * p['g1'])
            + p['s2'] * p['f2']**2 / (p['f2']**2 - f**2 - 1j * f * p['g2']))

Au_JC = mp.Medium(epsilon=_p['einf'], E_susceptibilities=[
    mp.DrudeSusceptibility(frequency=_p['f0'], gamma=_p['g0'], sigma=_p['s0']),
    mp.LorentzianSusceptibility(frequency=_p['f1'], gamma=_p['g1'], sigma=_p['s1']),
    mp.LorentzianSusceptibility(frequency=_p['f2'], gamma=_p['g2'], sigma=_p['s2']),
])


In [ ]:
%%writefile /content/sim/mie.py
"""Exact (Mie) decay rates of a radial dipole outside a sphere [Ruppin 1982; Kim et al. 1988].
Units: lengths in nm. eps_sphere(lam_nm) -> complex permittivity; n_host real."""
import numpy as np
from scipy.special import spherical_jn as jn, spherical_yn as yn

def _h(n, z, d=False):
    return jn(n, z, d) + 1j * yn(n, z, d)

def mie_an(n, x, m):
    mx = m * x
    psi = lambda z: z * jn(n, z)
    dpsi = lambda z: jn(n, z) + z * jn(n, z, True)
    xi = lambda z: z * _h(n, z)
    dxi = lambda z: _h(n, z) + z * _h(n, z, True)
    return (m * psi(mx) * dpsi(x) - psi(x) * dpsi(mx)) / (m * psi(mx) * dxi(x) - xi(x) * dpsi(mx))

def radial_dipole(lam_nm, R, d, eps_sphere, n_host=1.0, nmax=60):
    """Return (F_p, T) = (Gamma_tot/Gamma_0, Gamma_rad/Gamma_0) for a radial dipole at distance d."""
    Fp, T = [], []
    for lam in np.atleast_1d(lam_nm):
        k = 2 * np.pi * n_host / lam
        xs, x = k * R, k * (R + d)
        m = np.sqrt(eps_sphere(lam) + 0j) / n_host
        st = sr = 0
        for n in range(1, nmax):
            B = -mie_an(n, xs, m)
            c = n * (n + 1) * (2 * n + 1)
            st += c * B * (_h(n, x) / x) ** 2
            sr += c * abs(jn(n, x) + B * _h(n, x)) ** 2
        Fp.append(1 + 1.5 * np.real(st))
        T.append(1.5 / x**2 * sr)
    return np.array(Fp), np.array(T)


In [ ]:
%%writefile /content/sim/nanorod.py
"""Body-of-revolution FDTD (Meep, cylindrical coordinates) for a dipole near a gold nanorod or sphere.

Physics is fully 3D: the field is expanded as exp(i m phi); a dipole on the symmetry axis couples only to
m = 0 (axial / longitudinal dipole, Ez source) or m = +-1 (transverse dipole, Er source).
Outputs per wavelength:
    Fp   = P_tot / P_0         (total decay-rate enhancement, from the LDOS at the dipole)
    T    = P_rad / P_0         (radiated power through a closed box, normalised by the same box in free space)
    eta  = T / Fp              (apparent quantum efficiency for intrinsic yield q0 = 1)
    coll = fraction of radiated power inside a cone of numerical aperture NA around +z (optional)
Units: lengths in nm at the interface, Meep length unit = 1 um.
"""
import numpy as np
import meep as mp
from gold_jc import Au_JC

# Sub-bands: each gets its own Gaussian pulse; only the central part of each band is kept (band edges are noisy).
BANDS = [(0.46, 0.61), (0.54, 0.76), (0.63, 0.99)]
KEEP = [(0.49, 0.57), (0.57, 0.70), (0.70, 0.90)]


def geometry(shape, L=60.0, D=20.0, R_sph=15.874, material=Au_JC):
    """Rod: cylinder + two hemispherical caps, axis z, centre at origin. Returns (objects, z_top_apex)."""
    if shape == 'rod':
        r = D / 2000.0
        h = (L - D) / 1000.0
        objs = [mp.Cylinder(radius=r, height=h, center=mp.Vector3(), material=material),
                mp.Sphere(radius=r, center=mp.Vector3(0, 0, +h / 2), material=material),
                mp.Sphere(radius=r, center=mp.Vector3(0, 0, -h / 2), material=material)]
        return objs, L / 2000.0
    if shape == 'sphere':
        return [mp.Sphere(radius=R_sph / 1000.0, material=material)], R_sph / 1000.0
    raise ValueError(shape)


def _run_band(objs, z_top, gap, orient, res, n_host, band, nf, trun, dpml, pad, na_list):
    lo, hi = band
    fcen, df = 0.5 * (1 / lo + 1 / hi), 1 / lo - 1 / hi
    m = 0 if orient == 'z' else 1
    comp = mp.Ez if orient == 'z' else mp.Er
    zdip = z_top + gap / 1000.0
    half_z = max(abs(zdip), z_top) + pad          # flux box half-height
    rbox = max(z_top, 0.02) + pad                  # flux box radius (encloses rod or sphere)
    sr = rbox + 0.04
    sz = 2 * (half_z + 0.04)
    cell = mp.Vector3(sr + dpml, 0, sz + 2 * dpml)
    host = mp.Medium(index=n_host)
    out = {}
    for tag, geo in (('struct', objs), ('free', [])):
        sim = mp.Simulation(cell_size=cell, dimensions=mp.CYLINDRICAL, m=m, resolution=res * 1000,
                            boundary_layers=[mp.PML(dpml)], geometry=geo, default_material=host,
                            sources=[mp.Source(mp.GaussianSource(fcen, fwidth=df), component=comp,
                                               center=mp.Vector3(0, 0, zdip))])
        regs = [mp.FluxRegion(center=mp.Vector3(rbox / 2, 0, +half_z), size=mp.Vector3(rbox, 0, 0), weight=+1),
                mp.FluxRegion(center=mp.Vector3(rbox / 2, 0, -half_z), size=mp.Vector3(rbox, 0, 0), weight=-1),
                mp.FluxRegion(center=mp.Vector3(rbox, 0, 0), size=mp.Vector3(0, 0, 2 * half_z), weight=+1)]
        flux = sim.add_flux(fcen, df, nf, *regs)
        n2f = sim.add_near2far(fcen, df, nf, *[mp.Near2FarRegion(center=r.center, size=r.size, weight=r.weight) for r in regs]) if na_list else None
        sim.run(mp.dft_ldos(fcen, df, nf), until_after_sources=trun)
        rec = {'ldos': np.array(sim.ldos_data), 'flux': np.array(mp.get_fluxes(flux))}
        if n2f is not None:
            # far-field power density on a large circle in the r-z half plane; theta from +z axis
            th = np.linspace(0, np.pi, 181)
            Rff = 1000.0  # um, far zone
            S = []
            for t in th:
                ff = sim.get_farfield(n2f, mp.Vector3(Rff * np.sin(t), 0, Rff * np.cos(t)))
                E = np.array(ff).reshape(nf, 6)  # per frequency: (Er, Ep, Ez, Hr, Hp, Hz)
                e, h = E[:, :3], E[:, 3:]
                Sr = np.real(np.conj(e[:, 1]) * h[:, 2] - np.conj(e[:, 2]) * h[:, 1]) * np.sin(t) \
                     + np.real(np.conj(e[:, 0]) * h[:, 1] - np.conj(e[:, 1]) * h[:, 0]) * np.cos(t)
                S.append(Sr)
            rec['ff'] = (th, np.array(S))  # shape (n_theta, nf)
        out[tag] = rec
        sim.reset_meep()
    f = np.linspace(fcen - df / 2, fcen + df / 2, nf)
    lam = 1000.0 / f
    res_ = {'lam': lam, 'Fp': out['struct']['ldos'] / out['free']['ldos'],
            'T': out['struct']['flux'] / out['free']['flux']}
    if na_list:
        th, S = out['struct']['ff']
        w = np.sin(th)[:, None] * S                     # power per d(theta), axially symmetric
        tot = np.trapezoid(w, th, axis=0)
        for na in na_list:
            cut = np.arcsin(min(na / n_host, 1.0))
            msk = th <= cut
            res_[f'coll_NA{na}'] = np.trapezoid(w[msk], th[msk], axis=0) / tot
    return res_


def simulate(shape='rod', orient='z', gap=5.0, res=1.0, n_host=1.0, L=60.0, D=20.0, R_sph=15.874,
             nf=31, trun=60, dpml=0.3, pad=0.04, na_list=()):
    """Return dict of arrays over 490-900 nm. res = grid points per nm (1 -> 1 nm grid)."""
    objs, z_top = geometry(shape, L, D, R_sph)
    acc = {}
    for band, keep in zip(BANDS, KEEP):
        r = _run_band(objs, z_top, gap, orient, res, n_host, band, nf, trun, dpml, pad, na_list)
        m = (r['lam'] >= 1000 * keep[0]) & (r['lam'] <= 1000 * keep[1])
        for k, v in r.items():
            acc.setdefault(k, []).extend(list(np.asarray(v)[m]))
    o = np.argsort(acc['lam'])
    out = {k: np.asarray(v)[o] for k, v in acc.items()}
    out['eta'] = out['T'] / out['Fp']
    return out


def save_csv(path, out, meta=''):
    keys = ['lam', 'Fp', 'T', 'eta'] + sorted(k for k in out if k.startswith('coll'))
    np.savetxt(path, np.column_stack([out[k] for k in keys]), delimiter=',', header=','.join(keys),
               comments='# ' + meta + '\n')


In [ ]:
# 4) ابزار اجرای موازی: هر کار با mpirun روی همه‌ی هسته‌ها اجرا و نتیجه در Drive ذخیره می‌شود
import subprocess, os, sys, time, json, multiprocessing
NP = multiprocessing.cpu_count(); print('cores:', NP)
RUNNER = """
import sys, json; sys.path.insert(0, '/content/sim')
import meep as mp
from nanorod import simulate, save_csv
a = json.loads(sys.argv[1]); out = a.pop('out')
o = simulate(**a)
if mp.am_master(): save_csv(out, o, json.dumps(a))
"""
open('/content/sim/runner.py', 'w').write(RUNNER)
def job(name, **kw):
    path = f'{OUT}/{name}.csv'
    if os.path.exists(path):
        print(f'[skip] {name} (already done)'); return
    kw['out'] = path
    t = time.time(); print(f'[run ] {name} {kw}', flush=True)
    r = subprocess.run(['mpirun', '-np', str(NP), sys.executable, '/content/sim/runner.py', json.dumps(kw)],
                       capture_output=True, text=True)
    if r.returncode != 0 or not os.path.exists(path):
        print(r.stdout[-1500:], r.stderr[-1500:]); raise RuntimeError(name)
    print(f'[done] {name} in {(time.time()-t)/60:.1f} min', flush=True)

## مرحله‌ی A — کالیبراسیون: کره‌ی طلا در برابر حل دقیق می
اگر خطای میانه زیر حدود ۳٪ باشد، روش برای نانومیله قابل اعتماد است.

In [ ]:
import numpy as np
from mie import radial_dipole
from gold_jc import eps_jc
for res in [1.0, 2.0]:
    job(f'bench_sphere_res{res}', shape='sphere', orient='z', gap=5.0, res=res)
for res in [1.0, 2.0]:
    d = np.loadtxt(f'{OUT}/bench_sphere_res{res}.csv', delimiter=',')
    Fm, Tm = radial_dipole(d[:, 0], 15.874, 5.0, eps_jc)
    e = 100 * (d[:, 1] / Fm - 1)
    print(f'res {res}/nm: median |err Fp| {np.median(abs(e)):.1f}%  max {abs(e).max():.1f}%   peak Meep {d[:,1].max():.0f} vs Mie {Fm.max():.0f}')

## مرحله‌ی B — همگرایی مش برای حالت اصلی (میله، دوقطبی طولی، گاف ۵ nm) و حالت عرضی
شبکه‌ی ۱، ۰٫۶۷ و ۰٫۵ نانومتر. این همان آزمونی است که عدم‌قطعیت ±۴۰٪ مقاله را جایگزین می‌کند.

In [ ]:
for res in [1.0, 1.5, 2.0]:
    job(f'rodA_z_gap5_res{res}', shape='rod', orient='z', gap=5.0, res=res, na_list=[0.5, 0.9, 1.3])
for res in [1.0, 2.0]:
    job(f'rodC_x_gap5_res{res}', shape='rod', orient='x', gap=5.0, res=res)
    job(f'sphereDp_x_gap5_res{res}', shape='sphere', orient='x', gap=5.0, res=res)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for res in [1.0, 1.5, 2.0]:
    p = f'{OUT}/rodA_z_gap5_res{res}.csv'
    if not os.path.exists(p): continue
    d = np.loadtxt(p, delimiter=','); g = 1/res
    for j, a in enumerate(ax): a.plot(d[:, 0], d[:, j+1] * (100 if j == 2 else 1), label=f'grid {g:.2f} nm')
    i = d[:, 1].argmax(); k = d[:, 2].argmax()
    print(f'grid {g:.2f} nm: Fp {d[i,1]:.0f} @ {d[i,0]:.0f} nm | T {d[k,2]:.1f} @ {d[k,0]:.0f} nm | eta@T {100*d[k,3]:.1f}%')
for a, t in zip(ax, ['Fp', 'T', 'eta (%)']): a.set_title(t); a.set_xlabel('nm'); a.legend()
plt.show()

## مرحله‌ی C — نقشه‌های طراحی (سهم علمی تازه‌ی مقاله)
گاف × نسبت ابعاد، در هوا و آب، با شبکه‌ی ۱ نانومتر (`RES` را بعد از دیدن نتیجه‌ی مرحله‌ی B تنظیم کنید).
این مرحله طولانی است؛ هر زمان قطع شد، دوباره اجرا کنید تا از همان‌جا ادامه دهد.

In [ ]:
RES = 1.0
GAPS = [3, 5, 7, 10, 15, 20]
LENGTHS = [40, 50, 60, 80, 100]          # D = 20 nm  ->  aspect ratio 2, 2.5, 3, 4, 5
for n_host, tag in [(1.0, 'air'), (1.33, 'water')]:
    for L in LENGTHS:
        for g in GAPS:
            job(f'map_{tag}_L{L}_gap{g}_res{RES}', shape='rod', orient='z', gap=float(g), L=float(L), D=20.0,
                res=RES, n_host=n_host, na_list=[0.9, 1.2])

In [ ]:
# 5) بسته‌بندی همه‌ی نتایج برای ارسال
import shutil
shutil.make_archive('/content/results', 'zip', OUT)
from google.colab import files
files.download('/content/results.zip')